In [3]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 0s (31.6 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 121703 files and direc

# 데이터 불러오기

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='NanumBarunGothic')

!pip install tspoon
import tspoon as tsp

In [5]:
!pip install xmltodict
import pandas as pd

import requests
import xml.etree.ElementTree as ET
import xml.dom.minidom
import xmltodict

import json
from urllib.request import urlopen

def getECOS(topic,period,beg,end,item1='?',item2='?',item3='?',item4='?', df=None):
    key = ''
    url = 'https://ecos.bok.or.kr/api/StatisticSearch/'+'OIT9OYFOZDZWLHSBQJWU'+'/xml/kr/'+str(1)+'/'+str(50000)+'/'+topic+'/'+period+'/'+beg+'/'+end \
            +'/'+item1+'/'+item2+'/'+item3+'/'+item4
    # print(url)
    ## call OpenAPI URL
    response = requests.get(url)

    ## get API return value upon the http request
    if response.status_code == 200:
        try:
            contents = response.text
            ecosRoot = ET.fromstring(contents)

            # error check
            if ecosRoot[0].text[:4] in ("INFO","ERRO"):
                print(ecosRoot[0].text + " : " + ecosRoot[1].text)

            else:
                #print(contents)
                #dom = xml.dom.minidom.parse(xml_fname)
                dom = xml.dom.minidom.parseString(contents)
                pretty_xml_as_string = dom.toprettyxml(indent=" ")
                # print(pretty_xml_as_string)
                dic = xmltodict.parse(pretty_xml_as_string)

                n = int(dic['StatisticSearch']['list_total_count']['#text'])
                df_ecos = pd.DataFrame(index = [dic['StatisticSearch']['row'][i]['TIME'] for i in range(n)])
                df_ecos[dic['StatisticSearch']['row'][0]['ITEM_NAME1']] = [float(dic['StatisticSearch']['row'][i]['DATA_VALUE']) for i in range(n)]

                if type(df)==type(pd.DataFrame()):
                    df_ecos = df.merge(df_ecos, left_index=True, right_index=True, how='left')

                return df_ecos

        except Exception as e:
            print(str(e))


def getKOSIS(table,period,beg,end,item, orgId='101', obj1='',obj2='',obj3='',obj4='',obj5='',obj6='',obj7='',obj8='', title=None, title_no='0', debug=False):
     # Korean Statistics
    key = 'MzhiMmQwZTYyMWZiNGM1ZGFkZDJmZTI2OTE1OGU2NzQ='
    url = 'https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey='+key \
            +'&orgId='+orgId\
            +'&tblId='+table \
            +'&prdSe='+period \
            +'&startPrdDe='+beg \
            +'&endPrdDe='+end \
            +'&itmId='+item \
            +'&objL1='+obj1 \
            +'&objL2='+obj2 \
            +'&objL3='+obj3 \
            +'&objL4='+obj4 \
            +'&objL5='+obj5 \
            +'&objL6='+obj6 \
            +'&objL7='+obj7 \
            +'&objL8='+obj8 \
            +'&format=json&jsonVD=Y&outputFields=ITM_NM+PRD_DE+DT'
    res = requests.get(url)
    py_json = res.json()

    data = []
    for v in py_json:
        value = []
        value.append(v['PRD_DE'])

        # ✅ DT 값이 '-' 또는 ''일 때 NaN으로 처리
        try:
            value.append(float(v['DT']))
        except (ValueError, KeyError):
            value.append(np.nan)

        data.append(value)

    df = pd.DataFrame(data, columns=['PRD_DE', 'DT'])

    if debug:
        print(df.head())

    return df


    #get json data from url
    with urlopen(url) as url_:
        json_file = url_.read()

    py_json = json.loads(json_file.decode('utf-8'))


    #data transformation
    data = []

    for i, v in enumerate(py_json):
        value = []
        value.append(v['PRD_DE'])
        value.append(float(v['DT']))

        data.append(value)

    if title is not None:
        title = str(title)
    elif title_no=='0':
        title = v['ITM_NM_ENG']
    else:
        title = v['C'+title_no+'_NM_ENG']

    df_kosis = pd.DataFrame({v['PRD_SE']:[x[0] for x in data], title:[x[1] for x in data]}).set_index(v['PRD_SE'])

    if debug:
        return v

    return df_kosis

In [6]:
import pandas as pd
import numpy as np
import os
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

In [ ]:
# change directory to where you have your data file!!
%cd gdrive/MyDrive/GDP/데이터

## GDP


```
"('200Y104', 'Q', '1980Q1', '2025Q1','11055')"
"('200Y104', 'Q', '1980Q1', '2025Q1','11053')"
```


In [ ]:
gdp1 = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','11055').reset_index()
gdp2 = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','11053').reset_index()

In [ ]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [gdp1,gdp2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
gdp = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
gdp

gdp['gdp']=gdp['건물건설 및 건축보수업']+gdp['토목건설업']
gdp = gdp.drop(columns=['건물건설 및 건축보수업','토목건설업'])
gdp

##PPI


```
"('DT_39701_A003','M','200001','202506','16397AAA0','397','15397AA2AA1')"
"('DT_39701_A003','M','200001','202506','16397AAA0','397','15397AA2AA2')"
```


In [ ]:
ppi1 = getKOSIS('DT_39701_A003','M','201401','202506','16397AAA0','397','15397AA2AA1').reset_index()
ppi2 =getKOSIS('DT_39701_A003','M','201401','202506','16397AAA0','397','15397AA2AA2').reset_index()


In [ ]:
ppi1
ppi1 = ppi1.drop(columns=['index'])


In [ ]:
ppi1

In [ ]:
ppi2
ppi2 = ppi2.drop(columns=['index'])

In [ ]:
# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [ppi1,ppi2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
ppi = reduce(lambda left, right: pd.merge(left, right, on="PRD_DE", how="outer"), dfs)
ppi

In [ ]:
ppi['ppi']=(ppi['DT_x']+ppi['DT_y'])/2
ppi = ppi.drop(columns=["DT_x","DT_y"])
ppi

In [ ]:
ppi = ppi.rename(columns={'PRD_DE':'index'})

In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
ppi['index'] = ppi['index'].astype(str)

# 연도와 월 분리
ppi['연도'] = ppi['index'].str[:4].astype(int)
ppi['월'] = ppi['index'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
ppi['분기'] = ppi['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
ppi['index'] = ppi['연도'].astype(str) + ppi['분기']

# 분기별 평균 집계
ppi = ppi.groupby('index').mean(numeric_only=True).reset_index()


In [ ]:
ppi

In [ ]:
ppi = ppi.drop(columns=['연도', '월'], errors='ignore')

## BSI


```
"('512Y007','M','200908','202508','AA','F4100')","('512Y007','M','200908','202508','AB','F4100')","('512Y007','M','200908','202508','AE','F4100')","('512Y007','M','200908','202508','AO','F4100')","('512Y007','M','200908','202508','AJ','F4100')"
```


In [ ]:
업황실적BSI = getECOS('512Y007','M','201401','202506','AA','F4100').reset_index()
매출실적BSI = getECOS('512Y007','M','201401','202506','AB','F4100').reset_index()
채산성실적BSI = getECOS('512Y007','M','201401','202506','AE','F4100').reset_index()
자금사정실적BSI = getECOS('512Y007','M','201401','202506','AO','F4100').reset_index()
인력사정실적BSI= getECOS('512Y007','M','201401','202506','AJ','F4100').reset_index()

In [ ]:
업황실적BSI = 업황실적BSI.rename(columns={'\t업황실적BSI':'업황실적BSI'})
업황실적BSI

In [ ]:
매출실적BSI
매출실적BSI = 매출실적BSI.rename(columns={'매출실적BSI 2)':"매출실적BSI"})
매출실적BSI

In [ ]:
채산성실적BSI
채산성실적BSI = 채산성실적BSI.rename(columns={'채산성실적BSI 6)':"채산성실적BSI"})
채산성실적BSI

In [ ]:
자금사정실적BSI
자금사정실적BSI = 자금사정실적BSI.rename(columns={'자금사정실적BSI 6)':"자금사정실적BSI"})
자금사정실적BSI

In [ ]:
인력사정실적BSI
인력사정실적BSI = 인력사정실적BSI.rename(columns={'인력사정실적BSI 3)':"인력사정실적BSI"})
인력사정실적BSI

In [ ]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI, 매출실적BSI, 채산성실적BSI, 자금사정실적BSI, 인력사정실적BSI]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
df_merged = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
df_merged

In [ ]:
df_merged.isnull().sum()

In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
df_merged['index'] = df_merged['index'].astype(str)

# 연도와 월 분리
df_merged['연도'] = df_merged['index'].str[:4].astype(int)
df_merged['월'] = df_merged['index'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
df_merged['분기'] = df_merged['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
df_merged['index'] = df_merged['연도'].astype(str) + df_merged['분기']

# 분기별 평균 집계
df_quarter = df_merged.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(df_quarter.head())

In [ ]:
df_quarter = df_quarter.drop(columns=['연도', '월'], errors='ignore')

In [ ]:
dfs = [ppi, gdp]
df_final = df_quarter.copy()
key_col = 'index'
for df in dfs:
    df_final = df_final.merge(df, on=key_col, how='left')

df_final

,index,업황실적BSI 1),매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp
0,2014Q1,58.000000,72.333333,76.333333,74.333333,89.000000,84.916667,22695.5
1,2014Q2,58.666667,71.333333,78.000000,72.333333,87.000000,84.970000,22896.6
2,2014Q3,56.333333,73.666667,77.333333,75.666667,90.333333,85.076667,23225.2
3,2014Q4,58.666667,68.333333,76.333333,73.333333,94.333333,85.553333,23068.4
4,2015Q1,64.333333,71.000000,77.000000,80.666667,90.333333,85.571667,23282.7
5,2015Q2,67.333333,74.333333,81.666667,81.000000,86.666667,85.131667,23816.6
6,2015Q3,70.000000,77.000000,81.666667,79.666667,86.333333,85.191667,25057.8
7,2015Q4,69.666667,76.333333,81.000000,79.333333,86.333333,85.193333,25342.2
8,2016Q1,64.333333,75.000000,82.666667,79.666667,85.666667,85.026667,25704.7
9,2016Q2,65.000000,75.000000,83.333333,78.000000,82.000000,85.993333,26552.6


In [ ]:
df_final.to_csv("건설1차.csv")

## 생산지수

In [ ]:
생산지수 = getKOSIS('DT_1JH20202','Q','201401','202502','T1','101','1D')

In [ ]:
생산지수

,PRD_DE,DT
0,201401,80.4371
1,201402,82.3341
2,201403,80.0151
3,201404,79.0660
4,201501,81.1323
5,201502,81.2643
6,201503,86.8751
7,201504,87.5021
8,201601,90.2675
9,201602,94.0529


In [ ]:
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산지수['index']=생산지수['PRD_DE'].apply(to_quarter)
생산지수 = 생산지수.drop(columns=["PRD_DE"])
생산지수 = 생산지수.rename(columns={'DT': '생산지수'})
생산지수

,생산지수,index
0,80.4371,2014Q1
1,82.3341,2014Q2
2,80.0151,2014Q3
3,79.0660,2014Q4
4,81.1323,2015Q1
5,81.2643,2015Q2
6,86.8751,2015Q3
7,87.5021,2015Q4
8,90.2675,2016Q1
9,94.0529,2016Q2


## 건설수주액(총합)

https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=T10+&objL1=0+&objL2=0+&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Q&startPrdDe=201401&endPrdDe=202502&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=101&tblId=DT_1G1B002

In [ ]:
수주액 = getKOSIS('DT_1G1B002','Q','201401','202502','T10','101','0','0')

In [ ]:
수주액

,PRD_DE,DT
0,201401,16143968.0
1,201402,23621352.0
2,201403,24236310.0
3,201404,26603876.0
4,201501,25470011.0
5,201502,35381617.0
6,201503,36493946.0
7,201504,37147889.0
8,201601,28921640.0
9,201602,32789545.0


In [ ]:
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

수주액['index']=수주액['PRD_DE'].apply(to_quarter)
수주액 = 수주액.drop(columns=["PRD_DE"])
수주액 = 수주액.rename(columns={'DT': '수주액'})
수주액

,수주액,index
0,16143968.0,2014Q1
1,23621352.0,2014Q2
2,24236310.0,2014Q3
3,26603876.0,2014Q4
4,25470011.0,2015Q1
5,35381617.0,2015Q2
6,36493946.0,2015Q3
7,37147889.0,2015Q4
8,28921640.0,2016Q1
9,32789545.0,2016Q2


## 기업데이터

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('임지오기업.csv')

건물건설 및 건축보수업','토목건설업'

In [ ]:
건설기업1 = df[df['매핑한 산업'] == "건물건설 및 건축보수업"]
건설기업2 = df[df['매핑한 산업'] == "토목건설업"]

In [ ]:
columns = ['EBITDA', '인건비']
grouped1 = 건설기업1.groupby('조사일')[columns].sum()
grouped1

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,11721903.0
2000-04-01,0.000000e+00,23897306.0
2000-07-01,0.000000e+00,-1791384.0
2000-10-01,8.996119e+10,59173069.0
2001-01-01,0.000000e+00,25032573.0
...,...,...
2024-04-01,1.366543e+11,89805168.0
2024-07-01,5.218596e+10,87478058.0
2024-10-01,1.161212e+11,237928793.0


In [ ]:
grouped1 = grouped1.reset_index()

In [ ]:
grouped1['조사일']=pd.to_datetime(grouped1['조사일'])

# 분기(Quarter) 단위로 변환
grouped1["index"] = grouped1["조사일"].dt.to_period("Q")
grouped1 = grouped1.drop(columns=["조사일"])
print(grouped1)

grouped1

           EBITDA          인건비   index
0    0.000000e+00   11721903.0  2000Q1
1    0.000000e+00   23897306.0  2000Q2
2    0.000000e+00   -1791384.0  2000Q3
3    8.996119e+10   59173069.0  2000Q4
4    0.000000e+00   25032573.0  2001Q1
..            ...          ...     ...
97   1.366543e+11   89805168.0  2024Q2
98   5.218596e+10   87478058.0  2024Q3
99   1.161212e+11  237928793.0  2024Q4
100  2.177973e+11   72279583.0  2025Q1
101  1.772909e+11  -72279583.0  2025Q2

[102 rows x 3 columns]


,EBITDA,인건비,index
0,0.000000e+00,11721903.0,2000Q1
1,0.000000e+00,23897306.0,2000Q2
2,0.000000e+00,-1791384.0,2000Q3
3,8.996119e+10,59173069.0,2000Q4
4,0.000000e+00,25032573.0,2001Q1
...,...,...,...
97,1.366543e+11,89805168.0,2024Q2
98,5.218596e+10,87478058.0,2024Q3
99,1.161212e+11,237928793.0,2024Q4
100,2.177973e+11,72279583.0,2025Q1


In [ ]:
mask = (grouped1["index"] >= pd.Period("2014Q1")) & (grouped1["index"] <= pd.Period("2025Q2"))
기업1= grouped1.loc[mask]
기업1
기업1['합산'] = 기업1['EBITDA']+기업1['인건비']
기업1

기업1 = 기업1.rename(columns={'EBITDA': 'EBITDA1'})
기업1 = 기업1.rename(columns={'인건비': '인건비1'})
기업1 = 기업1.rename(columns={'합산': '합산1'})

In [ ]:
columns = ['EBITDA', '인건비']
grouped2 = 건설기업2.groupby('조사일')[columns].sum()
grouped2

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,1191356.0
2000-04-01,0.000000e+00,2199561.0
2000-07-01,0.000000e+00,190462.0
2000-10-01,1.439822e+11,147507232.0
2001-01-01,0.000000e+00,1944563.0
...,...,...
2024-04-01,1.606190e+11,32830870.0
2024-07-01,9.474970e+10,37183634.0
2024-10-01,1.075310e+11,394111783.0


In [ ]:
grouped2 = grouped2.reset_index()

In [ ]:
grouped2['조사일']=pd.to_datetime(grouped2['조사일'])

# 분기(Quarter) 단위로 변환
grouped2["index"] = grouped2["조사일"].dt.to_period("Q")
grouped2 = grouped2.drop(columns=["조사일"])
print(grouped2)

grouped2

           EBITDA          인건비   index
0    0.000000e+00    1191356.0  2000Q1
1    0.000000e+00    2199561.0  2000Q2
2    0.000000e+00     190462.0  2000Q3
3    1.439822e+11  147507232.0  2000Q4
4    0.000000e+00    1944563.0  2001Q1
..            ...          ...     ...
97   1.606190e+11   32830870.0  2024Q2
98   9.474970e+10   37183634.0  2024Q3
99   1.075310e+11  394111783.0  2024Q4
100  5.668363e+10   15621169.0  2025Q1
101  1.488576e+11  -15621169.0  2025Q2

[102 rows x 3 columns]


,EBITDA,인건비,index
0,0.000000e+00,1191356.0,2000Q1
1,0.000000e+00,2199561.0,2000Q2
2,0.000000e+00,190462.0,2000Q3
3,1.439822e+11,147507232.0,2000Q4
4,0.000000e+00,1944563.0,2001Q1
...,...,...,...
97,1.606190e+11,32830870.0,2024Q2
98,9.474970e+10,37183634.0,2024Q3
99,1.075310e+11,394111783.0,2024Q4
100,5.668363e+10,15621169.0,2025Q1


In [ ]:
mask = (grouped2["index"] >= pd.Period("2014Q1")) & (grouped2["index"] <= pd.Period("2025Q2"))
기업2= grouped2.loc[mask]
기업2
기업2['합산'] = 기업2['EBITDA']+기업2['인건비']
기업2

기업2 = 기업2.rename(columns={'EBITDA': 'EBITDA2'})
기업2 = 기업2.rename(columns={'인건비': '인건비2'})
기업2 = 기업2.rename(columns={'합산': '합산2'})

In [ ]:
# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [기업1,기업2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
기업 = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
기업

기업['EBITDA']=기업['EBITDA1']+기업['EBITDA2']
기업['인건비']=기업['인건비1']+기업['인건비2']
기업['합산']=기업['합산1']+기업['합산2']
기업

,EBITDA1,인건비1,index,합산1,EBITDA2,인건비2,합산2,EBITDA,인건비,합산
0,1.637860e+11,6.571266e+07,2014Q1,1.638517e+11,6.770055e+10,2.138264e+07,6.772193e+10,2.314865e+11,8.709530e+07,2.315736e+11
1,1.701119e+11,5.789741e+07,2014Q2,1.701698e+11,8.742687e+10,2.405856e+07,8.745093e+10,2.575388e+11,8.195597e+07,2.576207e+11
2,1.306489e+11,6.830529e+07,2014Q3,1.307172e+11,6.493368e+10,2.109998e+07,6.495478e+10,1.955826e+11,8.940527e+07,1.956720e+11
3,1.450417e+11,1.059982e+08,2014Q4,1.451477e+11,1.471675e+11,4.753564e+08,1.476428e+11,2.922092e+11,5.813547e+08,2.927906e+11
4,1.169975e+11,7.227285e+07,2015Q1,1.170698e+11,6.233967e+10,2.237752e+07,6.236205e+10,1.793372e+11,9.465036e+07,1.794318e+11
5,1.488570e+11,5.826302e+07,2015Q2,1.489152e+11,1.046576e+11,2.729910e+07,1.046849e+11,2.535145e+11,8.556212e+07,2.536001e+11
6,1.884142e+11,7.616550e+07,2015Q3,1.884904e+11,9.225594e+10,2.580023e+07,9.228174e+10,2.806702e+11,1.019657e+08,2.807721e+11
7,-1.794616e+10,9.464391e+08,2015Q4,-1.699972e+10,1.176698e+11,9.456737e+08,1.186154e+11,9.972361e+10,1.892113e+09,1.016157e+11
8,1.514887e+11,7.069080e+07,2016Q1,1.515594e+11,7.648992e+10,2.786320e+07,7.651778e+10,2.279786e+11,9.855400e+07,2.280772e+11
9,2.057817e+11,6.518925e+07,2016Q2,2.058469e+11,1.005254e+11,2.811255e+07,1.005535e+11,3.063071e+11,9.330181e+07,3.064004e+11


In [ ]:
기업 = 기업.drop(columns=["EBITDA1","EBITDA2","인건비1","인건비2","합산1","합산2"])
기업

,index,EBITDA,인건비,합산
0,2014Q1,2.314865e+11,8.709530e+07,2.315736e+11
1,2014Q2,2.575388e+11,8.195597e+07,2.576207e+11
2,2014Q3,1.955826e+11,8.940527e+07,1.956720e+11
3,2014Q4,2.922092e+11,5.813547e+08,2.927906e+11
4,2015Q1,1.793372e+11,9.465036e+07,1.794318e+11
5,2015Q2,2.535145e+11,8.556212e+07,2.536001e+11
6,2015Q3,2.806702e+11,1.019657e+08,2.807721e+11
7,2015Q4,9.972361e+10,1.892113e+09,1.016157e+11
8,2016Q1,2.279786e+11,9.855400e+07,2.280772e+11
9,2016Q2,3.063071e+11,9.330181e+07,3.064004e+11


## 전체 데이터 합산

In [ ]:
from functools import reduce
import pandas as pd

# 모든 데이터프레임의 index 컬럼을 문자열로 변환
for df in [df_final,생산지수,수주액,기업]:
    df['index'] = df['index'].astype(str)

# 리스트에 넣고 합치기
dfs = [df_final,생산지수,수주액,기업]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='index', how='outer'), dfs)


In [ ]:
df_merged
df_merged = df_merged[['index'] + [col for col in df_merged.columns if col != 'index']]

In [ ]:
df_merged

,index,업황실적BSI 1),매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,수주액,EBITDA,인건비,합산
0,2014Q1,58.000000,72.333333,76.333333,74.333333,89.000000,84.916667,22695.5,80.4371,16143968.0,2.314865e+11,8.709530e+07,2.315736e+11
1,2014Q2,58.666667,71.333333,78.000000,72.333333,87.000000,84.970000,22896.6,82.3341,23621352.0,2.575388e+11,8.195597e+07,2.576207e+11
2,2014Q3,56.333333,73.666667,77.333333,75.666667,90.333333,85.076667,23225.2,80.0151,24236310.0,1.955826e+11,8.940527e+07,1.956720e+11
3,2014Q4,58.666667,68.333333,76.333333,73.333333,94.333333,85.553333,23068.4,79.0660,26603876.0,2.922092e+11,5.813547e+08,2.927906e+11
4,2015Q1,64.333333,71.000000,77.000000,80.666667,90.333333,85.571667,23282.7,81.1323,25470011.0,1.793372e+11,9.465036e+07,1.794318e+11
5,2015Q2,67.333333,74.333333,81.666667,81.000000,86.666667,85.131667,23816.6,81.2643,35381617.0,2.535145e+11,8.556212e+07,2.536001e+11
6,2015Q3,70.000000,77.000000,81.666667,79.666667,86.333333,85.191667,25057.8,86.8751,36493946.0,2.806702e+11,1.019657e+08,2.807721e+11
7,2015Q4,69.666667,76.333333,81.000000,79.333333,86.333333,85.193333,25342.2,87.5021,37147889.0,9.972361e+10,1.892113e+09,1.016157e+11
8,2016Q1,64.333333,75.000000,82.666667,79.666667,85.666667,85.026667,25704.7,90.2675,28921640.0,2.279786e+11,9.855400e+07,2.280772e+11
9,2016Q2,65.000000,75.000000,83.333333,78.000000,82.000000,85.993333,26552.6,94.0529,32789545.0,3.063071e+11,9.330181e+07,3.064004e+11


In [ ]:
df_merged.to_csv('건설.csv')

# 데이터 QoQ 변환

In [ ]:
import pandas as pd

# df는 위의 데이터프레임
df_qoq = df_merged.copy()

# 분기 순서대로 정렬 (혹시 index가 문자열일 경우를 대비)
df_qoq = df_qoq.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_qoq.select_dtypes(include=['number']).columns

# 전분기 대비 증감률 (%) 계산
df_qoq[num_cols] = df_qoq[num_cols].pct_change() * 100

# 결과 확인
print(df_qoq.head())

    index  업황실적BSI 1)   매출실적BSI  채산성실적BSI  자금사정실적BSI  인력사정실적BSI       ppi  \
0  2014Q1         NaN       NaN       NaN        NaN        NaN       NaN   
1  2014Q2    1.149425 -1.382488  2.183406  -2.690583  -2.247191  0.062807   
2  2014Q3   -3.977273  3.271028 -0.854701   4.608295   3.831418  0.125535   
3  2014Q4    4.142012 -7.239819 -1.293103  -3.083700   4.428044  0.560279   
4  2015Q1    9.659091  3.902439  0.873362  10.000000  -4.240283  0.021429   

        gdp      생산지수        수주액     EBITDA         인건비         합산  
0       NaN       NaN        NaN        NaN         NaN        NaN  
1  0.886079  2.358364  46.316891  11.254322   -5.900818  11.247870  
2  1.435148 -2.816573   2.603399 -24.057026    9.089395 -24.046482  
3 -0.675129 -1.186151   9.768674  49.404517  550.246541  49.633358  
4  0.928976  2.613386  -4.262029 -38.627136  -83.718998 -38.716669  


In [ ]:
df_qoq
df_qoq = df_qoq.drop(index=[0, 1, 2,3])

In [ ]:
df_qoq.isnull().sum()

,0
index,0
업황실적BSI 1),0
매출실적BSI,0
채산성실적BSI,0
자금사정실적BSI,0
인력사정실적BSI,0
ppi,0
gdp,0
생산지수,0
수주액,0


In [ ]:
df_qoq.to_csv('건설qoq.csv')

## 데이터 YoY 변환

In [ ]:
import pandas as pd

# df_merged는 원본 데이터프레임
df_yoy = df_merged.copy()

# 분기 순서대로 정렬
df_yoy = df_yoy.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_yoy.select_dtypes(include=['number']).columns

# 전년 동분기 대비 증감률 (%) 계산 (4분기 차이)
df_yoy[num_cols] = df_yoy[num_cols].pct_change(periods=4) * 100

# 결과 확인
print(df_yoy.head(10))

    index  업황실적BSI 1)    매출실적BSI  채산성실적BSI  자금사정실적BSI  인력사정실적BSI       ppi  \
0  2014Q1         NaN        NaN       NaN        NaN        NaN       NaN   
1  2014Q2         NaN        NaN       NaN        NaN        NaN       NaN   
2  2014Q3         NaN        NaN       NaN        NaN        NaN       NaN   
3  2014Q4         NaN        NaN       NaN        NaN        NaN       NaN   
4  2015Q1   10.919540  -1.843318  0.873362   8.520179   1.498127  0.771344   
5  2015Q2   14.772727   4.205607  4.700855  11.981567  -0.383142  0.190263   
6  2015Q3   24.260355   4.524887  5.603448   5.286344  -4.428044  0.135172   
7  2015Q4   18.750000  11.707317  6.113537   8.181818  -8.480565 -0.420790   
8  2016Q1    0.000000   5.633803  7.359307  -1.239669  -5.166052 -0.636893   
9  2016Q2   -3.465347   0.896861  2.040816  -3.703704  -5.384615  1.012158   

         gdp       생산지수        수주액     EBITDA         인건비         합산  
0        NaN        NaN        NaN        NaN         NaN        NaN  

In [ ]:
df_yoy=df_yoy.dropna()

In [ ]:
df_yoy

,index,업황실적BSI 1),매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,수주액,EBITDA,인건비,합산
4,2015Q1,10.919540,-1.843318,0.873362,8.520179,1.498127,0.771344,2.587297,0.864278,57.767973,-22.528029,8.674477,-22.516294
5,2015Q2,14.772727,4.205607,4.700855,11.981567,-0.383142,0.190263,4.018064,-1.299340,49.786587,-1.562571,4.400107,-1.560674
6,2015Q3,24.260355,4.524887,5.603448,5.286344,-4.428044,0.135172,7.890567,8.573382,50.575504,43.504674,14.048908,43.491215
7,2015Q4,18.750000,11.707317,6.113537,8.181818,-8.480565,-0.420790,9.856774,10.669694,39.633371,-65.872531,225.466173,-65.294059
8,2016Q1,0.000000,5.633803,7.359307,-1.239669,-5.166052,-0.636893,10.402574,11.259634,13.551737,27.122912,4.124267,27.110780
9,2016Q2,-3.465347,0.896861,2.040816,-3.703704,-5.384615,1.012158,11.487786,15.737046,-7.326042,20.824266,9.045696,20.820292
10,2016Q3,3.809524,6.060606,1.632653,2.092050,-1.930502,1.657048,10.618251,14.998083,0.588731,6.123619,8.833955,6.124603
11,2016Q4,-2.392344,9.170306,5.761317,6.302521,2.316602,3.136004,10.433190,20.258485,26.429219,-598.851845,-11.924179,-587.923090
12,2017Q1,3.108808,8.000000,-0.403226,-1.673640,1.556420,5.923632,11.742989,18.304595,-1.094316,90.082047,-1.160483,90.042621
13,2017Q2,9.743590,12.888889,-2.000000,-0.854701,3.658537,4.515854,7.918999,15.351573,18.264111,64.157953,4.503451,64.139788


In [ ]:
df_yoy.to_csv("건설yoy.csv")

# 상관계수 보기

In [ ]:
qoq = pd.read_csv('건설qoq.csv')
qoq = qoq.drop(columns=["Unnamed: 0"])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# GDP_QoQ 기준
target = 'gdp'

In [ ]:
# 수치형 컬럼만 선택
num_cols = qoq.select_dtypes(include=['number']).columns

# GDP 기준 상관계수 계산
corr = qoq[num_cols].corr()[target].sort_values(ascending=False)

print("📊 GDP 대비 단순 상관계수")
print(corr)


📊 GDP 대비 단순 상관계수
gdp           1.000000
생산지수          0.748036
매출실적BSI       0.066312
EBITDA        0.045356
합산            0.042942
ppi           0.030663
자금사정실적BSI     0.020897
업황실적BSI 1)   -0.013753
인건비          -0.030281
인력사정실적BSI    -0.080719
채산성실적BSI     -0.133615
수주액          -0.285683
Name: gdp, dtype: float64


In [ ]:
# df2 사본
df2 = qoq.copy()

# target 및 feature 지정
target_col = 'gdp'
feature_cols = [col for col in df2.columns if col not in ['index', target_col]]

# 결과 저장용 데이터프레임
corr_df = pd.DataFrame(columns=['Feature', 'Lag', 'Correlation'])

# lag 1~4 생성 및 상관계수 계산
for col in feature_cols:
    for lag in range(1, 5):  # lag 1~4
        lag_col_name = f"{col}_lag{lag}"
        df2[lag_col_name] = df2[col].shift(lag)  # 실제 데이터에 lag 컬럼 추가
        corr = df2[target_col].corr(df2[lag_col_name])  # 상관계수 계산
        corr_df = pd.concat(
            [corr_df, pd.DataFrame({'Feature': [col], 'Lag': [lag], 'Correlation': [corr]})],
            ignore_index=True
        )

# 상관계수 높은 순으로 정렬
corr_df.sort_values(by='Correlation', ascending=False, inplace=True)
corr_df.reset_index(drop=True, inplace=True)

corr_df

,Feature,Lag,Correlation
0,매출실적BSI,1,0.347735
1,수주액,1,0.344052
2,매출실적BSI,2,0.313936
3,생산지수,2,0.279055
4,채산성실적BSI,2,0.250601
5,업황실적BSI 1),3,0.244021
6,업황실적BSI 1),1,0.215692
7,인건비,1,0.191717
8,자금사정실적BSI,2,0.189087
9,생산지수,4,0.184263


In [ ]:
df2

,index,업황실적BSI 1),매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,수주액,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
0,2015Q1,9.659091,3.902439,0.873362,10.000000,-4.240283,0.021429,0.928976,2.613386,-4.262029,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015Q2,4.663212,4.694836,6.060606,0.413223,-4.059041,-0.514189,2.293119,0.162697,38.914808,...,NaN,NaN,-83.718998,NaN,NaN,NaN,-38.716669,NaN,NaN,NaN
2,2015Q3,3.960396,3.587444,0.000000,-1.646091,-0.384615,0.070479,5.211491,6.904385,3.143805,...,NaN,NaN,-9.601914,-83.718998,NaN,NaN,41.335074,-38.716669,NaN,NaN
3,2015Q4,-0.476190,-0.865801,-0.816327,-0.418410,0.000000,0.001956,1.134976,0.721726,1.791922,...,-38.627136,NaN,19.171587,-9.601914,-83.718998,NaN,10.714519,41.335074,-38.716669,NaN
4,2016Q1,-7.655502,-1.746725,2.057613,0.420168,-0.772201,-0.195633,1.430420,3.160381,-22.144593,...,41.361957,-38.627136,1755.635946,19.171587,-9.601914,-83.718998,-63.808471,10.714519,41.335074,-38.716669
5,2016Q2,1.036269,0.000000,0.806452,-2.092050,-4.280156,1.136898,3.298619,4.193536,13.373740,...,10.711665,41.361957,-94.791325,1755.635946,19.171587,-9.601914,124.450678,-63.808471,10.714519,41.335074
6,2016Q3,11.794872,8.888889,-0.400000,4.273504,3.252033,0.709357,4.390907,6.221818,11.952749,...,-64.469464,10.711665,-5.329253,-94.791325,1755.635946,19.171587,34.340651,124.450678,-63.808471,10.714519
7,2016Q4,-6.422018,2.040816,3.212851,3.688525,4.330709,1.456834,0.965781,5.329079,27.941501,...,128.610482,-64.469464,18.940184,-5.329253,-94.791325,1755.635946,-2.751978,34.340651,124.450678,-63.808471
8,2017Q1,-2.450980,-2.800000,-3.891051,-7.114625,-1.509434,2.501944,2.633441,1.484291,-39.093650,...,34.357800,128.610482,1401.706518,18.940184,-5.329253,-94.791325,-266.395748,-2.751978,34.340651,124.450678
9,2017Q2,7.537688,4.526749,-0.809717,-1.276596,-2.298851,-0.207262,-0.236394,1.592743,35.563944,...,-2.758586,34.357800,-94.154776,1401.706518,18.940184,-5.329253,-187.421965,-266.395748,-2.751978,34.340651


In [ ]:
# 숫자형 컬럼만 선택 (연도분기 제외)
numeric_df = df2.select_dtypes(include=["number"])

# 전체 상관계수
corr = numeric_df.corr()

target_corr = corr["gdp"].drop("gdp")
target_corr = target_corr.sort_values(ascending=False)
target_corr

,gdp
생산지수,0.748036
매출실적BSI_lag1,0.347735
수주액_lag1,0.344052
매출실적BSI_lag2,0.313936
생산지수_lag2,0.279055
채산성실적BSI_lag2,0.250601
업황실적BSI 1)_lag3,0.244021
업황실적BSI 1)_lag1,0.215692
인건비_lag1,0.191717
자금사정실적BSI_lag2,0.189087


In [ ]:
target_corr.to_csv('건설순서.csv')

In [ ]:
df2.dropna(inplace=True)

In [ ]:
df2

,index,업황실적BSI 1),매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,수주액,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
4,2016Q1,-7.655502,-1.746725,2.057613,0.420168,-0.772201,-0.195633,1.430420,3.160381,-22.144593,...,41.361957,-38.627136,1755.635946,19.171587,-9.601914,-83.718998,-63.808471,10.714519,41.335074,-38.716669
5,2016Q2,1.036269,0.000000,0.806452,-2.092050,-4.280156,1.136898,3.298619,4.193536,13.373740,...,10.711665,41.361957,-94.791325,1755.635946,19.171587,-9.601914,124.450678,-63.808471,10.714519,41.335074
6,2016Q3,11.794872,8.888889,-0.400000,4.273504,3.252033,0.709357,4.390907,6.221818,11.952749,...,-64.469464,10.711665,-5.329253,-94.791325,1755.635946,19.171587,34.340651,124.450678,-63.808471,10.714519
7,2016Q4,-6.422018,2.040816,3.212851,3.688525,4.330709,1.456834,0.965781,5.329079,27.941501,...,128.610482,-64.469464,18.940184,-5.329253,-94.791325,1755.635946,-2.751978,34.340651,124.450678,-63.808471
8,2017Q1,-2.450980,-2.800000,-3.891051,-7.114625,-1.509434,2.501944,2.633441,1.484291,-39.093650,...,34.357800,128.610482,1401.706518,18.940184,-5.329253,-94.791325,-266.395748,-2.751978,34.340651,124.450678
9,2017Q2,7.537688,4.526749,-0.809717,-1.276596,-2.298851,-0.207262,-0.236394,1.592743,35.563944,...,-2.758586,34.357800,-94.154776,1401.706518,18.940184,-5.329253,-187.421965,-266.395748,-2.751978,34.340651
10,2017Q3,1.401869,-2.362205,4.081633,4.310345,-1.568627,0.702815,1.852711,4.700276,-13.133293,...,-267.017236,-2.758586,0.095792,-94.154776,1401.706518,18.940184,16.030004,-187.421965,-266.395748,-2.751978
11,2017Q4,0.000000,6.048387,1.568627,8.264463,2.390438,1.743854,-2.207893,-5.950741,24.379327,...,-187.109527,-267.017236,0.100158,0.095792,-94.154776,1401.706518,-40.979163,16.030004,-187.421965,-266.395748
12,2018Q1,-2.764977,-6.844106,-8.108108,-5.343511,3.891051,1.596322,1.782649,2.589782,-18.283690,...,16.033586,-187.109527,2171.869094,0.100158,0.095792,-94.154776,-69.694468,-40.979163,16.030004,-187.421965
13,2018Q2,6.161137,-4.489796,5.042017,0.403226,-6.741573,0.334913,-1.912869,-3.580628,0.054333,...,-40.987128,16.033586,-95.618994,2171.869094,0.100158,0.095792,291.449440,-69.694468,-40.979163,16.030004


In [ ]:
df2 = df2.reset_index()
df2 = df2.drop(columns=["level_0"])

In [ ]:
df2.to_csv('건설lag.csv', encoding='utf-8-sig')

# ARIMA

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('건설lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','매출실적BSI_lag1','수주액_lag1']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (1.0, 1.0, 1.0)
📉 평균 Train MAE: 1.2177
📈 평균 Test MAE: 1.4015


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('건설lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['생산지수','매출실적BSI_lag1']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (1.0, 1.0, 1.0)
📉 평균 Train MAE: 1.2172
📈 평균 Test MAE: 1.6042


## 아리마 예측값

In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('건설lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','매출실적BSI_lag1','수주액_lag1']

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    pred = fit.forecast(steps=1, exog=X_test)

                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 최적 (p,d,q) 선택
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

best_p, best_d, best_q = int(best['p']), int(best['d']), int(best['q'])

print("✅ 최적 ARIMAX(p,d,q):", (best_p, best_d, best_q))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

# ---------------------------------------------------
# 8️⃣ 최적 (p,d,q)로 전체 구간 롤링 예측값 저장
# ---------------------------------------------------
rolling_preds = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(train_data) < train_window or len(X_test) == 0:
        continue

    try:
        model = SARIMAX(train_data, exog=X_train, order=(best_p, best_d, best_q),
                        enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        pred = fit.forecast(steps=1, exog=X_test)

        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(y.loc[test_end])
        })
    except:
        continue

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 최적 모델 예측값(pred) 시계열:")
print(pred_df)


FileNotFoundError: [Errno 2] No such file or directory: '건설lag.csv'

In [ ]:
level_2025Q1 = 12065.9

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

# 랜덤포레스트

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('건설lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','매출실적BSI_lag1','수주액_lag1'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

# 첫 번째 윈도우 구간 설정
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# 하이퍼파라미터 탐색 (단 1회)
rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,            # 108조합 중 10개만 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'n_estimators': 400, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 0.9353
📈 평균 Test MAE: 2.0062


# AR

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('건설qoq.csv')

# GDP 단일 시계열만 사용 (AR(1))
y = df['gdp'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train 예측 ---
        # fittedvalues는 길이가 n-1이므로 첫 번째 값 제외하고 MAE 계산
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 (1분기 앞 예측) ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 1.5900
📈 평균 Test MAE: 2.0278


#XGB

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('건설lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','매출실적BSI_lag1','수주액_lag1'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# 첫 번째 윈도우로 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# RandomizedSearchCV 실행
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,          # 전체 중 15조합 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'subsample': 1.0, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 1.0}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.1482
📈 평균 Test MAE: 1.9354


#MLP

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('건설lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','매출실적BSI_lag1','수주액_lag1'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# 스케일링 (NN에서는 매우 중요)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16), (64,32), (128,64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

# 첫 번째 윈도우로 하이퍼파라미터 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=1000)
rand_search = RandomizedSearchCV(
    mlp,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y_scaled.loc[train_start:train_end]
    y_test = y_scaled.loc[test_end:test_end]
    X_train = X_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = MLPRegressor(**best_params, random_state=42, max_iter=1000)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred.reshape(-1, 1)).flatten()
        test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1)).flatten()
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (64,), 'alpha': 0.0001, 'activation': 'tanh'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.8186
📈 평균 Test MAE: 2.8288


#LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('건설lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['생산지수','매출실적BSI_lag1','수주액_lag1'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 4️⃣ LSTM 입력 형태 변환 함수
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    """
    lookback: 과거 몇 분기 데이터를 사용할지 (ex. 4 = 1년)
    """
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ LSTM 모델 정의 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측
# -----------------------------
train_mae_list, test_mae_list = [], []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    # 윈도우 범위 맞추기
    train_mask = (np.array(idxs) >= train_start) & (np.array(idxs) <= train_end)
    test_mask = (np.array(idxs) == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test, y_test = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        model.fit(X_train, y_train, epochs=200, batch_size=8, verbose=0, callbacks=[early_stop])

        # 예측
        train_pred = model.predict(X_train, verbose=0)
        test_pred = model.predict(X_test, verbose=0)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1))
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1))

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 8️⃣ 결과 요약
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 0.8344
📈 평균 Test MAE: 2.9217
